# 🎙️ Breeze-TTS-2

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

A speech synthesis notebook powered by [BreezeBlue/Breeze-TTS-2](https://huggingface.co/BreezeBlue/Breeze-TTS-2).

### 🏛️ Core Engine Modes
- **🎨 Voice Design**: Zero-shot voice creation from natural-language descriptions without reference audio.
- **🧬 Voice Clone**: Exact timbre and acoustic cloning with Whisper auto-transcription.
- **🎛️ Voice Direction**: Speaker identity preservation with emotional & pacing steering.
- **🎭 Inline Vocal Events**: Interactive one-click insertions for English `(laugh)`, `(sigh)` and Chinese `[笑]`, `[叹气]`.

> ⚠️ **Runtime Requirement**: Set runtime to GPU (**Runtime → Change runtime type → T4 GPU**).

In [ ]:
#@title Step 1: Install System Tools & Tokenizer Dependencies (< 45 seconds)
import os
import subprocess

print("🚀 Installing system packages, audio codecs, and tokenizer dependencies...")

# 1. Install localtunnel and aria2c
!apt-get update -qq && apt-get install -y -qq aria2
!npm install -g -q localtunnel
!pip install -q uv

# 2. Clone official inference codebase
if not os.path.exists("/content/breeze-tts"):
    !git clone -q https://github.com/breezeblue-ai/breeze-tts.git /content/breeze-tts

# 3. Install dependencies via UV for optimized execution speed
!uv pip install --system --quiet sox flask flask-cors pydub huggingface_hub soundfile openai-whisper qwen-tts einops

# 4. Remove incompatible torchvision binary to prevent 'torchvision::nms does not exist'
!pip uninstall -y -q torchvision

print("✅ Setup finished successfully in under a minute!")

In [ ]:
#@title Step 2: Download Model Weights & Apply Safe Patches
import os
import re
from huggingface_hub import snapshot_download

%cd /content/breeze-tts
# 1. Reset all repo files to clean state
!git reset --hard HEAD

MODEL_DIR = "/content/Breeze-TTS-2"
os.makedirs("/content/outputs", exist_ok=True)
os.makedirs("/content/uploads", exist_ok=True)

# 2. Patch AutoConfig.register to allow exist_ok=True
cfg_path = "/content/breeze-tts/models/breeze_config.py"
if os.path.exists(cfg_path):
    with open(cfg_path, "r") as f:
        cfg = f.read()
    cfg = re.sub(r'AutoConfig\.register\((.*?)\)', lambda m: m.group(0) if 'exist_ok' in m.group(1) else f"AutoConfig.register({m.group(1)}, exist_ok=True)", cfg)
    with open(cfg_path, "w") as f:
        f.write(cfg)

# 3. Inject Transformers 5+ no_init_weights bridge into infer.py
infer_path = "/content/breeze-tts/infer.py"
shim = '''
import transformers.modeling_utils
try:
    from transformers.initialization import no_init_weights
    transformers.modeling_utils.no_init_weights = no_init_weights
except ImportError:
    from contextlib import contextmanager
    @contextmanager
    def _dummy(*a, **k): yield
    transformers.modeling_utils.no_init_weights = _dummy
'''
with open(infer_path, "r") as f:
    inf = f.read()

if "from __future__ import annotations" in inf:
    inf = inf.replace("from __future__ import annotations", "from __future__ import annotations\n" + shim, 1)
else:
    inf = shim + "\n" + inf

with open(infer_path, "w") as f:
    f.write(inf)

print("✅ Compatibility patches applied.")

# 4. Accelerated download of Breeze-TTS-2 weights (~7.7 GB)
print("📥 Downloading Breeze-TTS-2 checkpoint from Hugging Face via fast transfer...")
snapshot_download(
    repo_id="BreezeBlue/Breeze-TTS-2",
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False
)
print("✅ Model checkpoint ready at /content/Breeze-TTS-2!")

In [ ]:
#@title Step 3A: Write Server File
%%writefile /content/breeze-tts/app.py
import os
import sys
import re
import subprocess
import uuid
import traceback
import threading
from flask import Flask, render_template_string, request, send_from_directory, jsonify
from flask_cors import CORS
from pydub import AudioSegment
import whisper
import torch

app = Flask(__name__)
CORS(app)

OUTPUT_DIR = "/content/outputs"
UPLOAD_DIR = "/content/uploads"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("[INIT] Loading Whisper model for voice clone auto-transcription...", flush=True)
whisper_model = whisper.load_model("base")
print("[INIT] Whisper model ready.", flush=True)

# Pre-import inference engine to optimize execution without subprocessing
sys.path.append("/content/breeze-tts")
try:
    from infer import BreezeTTSInference
    print("[INIT] Pre-loading Breeze-TTS-2 Engine directly into VRAM...", flush=True)
    tts_engine = BreezeTTSInference("/content/Breeze-TTS-2")
    print("[INIT] Breeze-TTS-2 Engine pre-loaded successfully!", flush=True)
except Exception as e:
    print(f"[INIT WARNING] Direct engine import failed, falling back to CLI execution: {e}", flush=True)
    tts_engine = None

tasks = {}

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Breeze-TTS-2</title>
    <style>
        :root {
            --bg: #0b1329;
            --card: #152243;
            --accent: #38bdf8;
            --accent-glow: rgba(56, 189, 248, 0.25);
            --text: #f8fafc;
            --subtext: #94a3b8;
            --border: #233563;
            --tag-bg: #1e293b;
            --error: #f87171;
            --success: #4ade80;
        }
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
            background-color: var(--bg);
            color: var(--text);
            margin: 0;
            padding: 1.5rem;
            display: flex;
            justify-content: center;
        }
        .container {
            max-width: 1100px;
            width: 100%;
            background: var(--card);
            border-radius: 14px;
            padding: 2rem;
            box-shadow: 0 15px 35px -5px rgba(0,0,0,0.6);
            border: 1px solid var(--border);
        }
        h1 {
            margin-top: 0;
            font-size: 1.8rem;
            color: var(--accent);
            border-bottom: 1px solid var(--border);
            padding-bottom: 0.8rem;
            margin-bottom: 0.4rem;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }
        .attribution {
            font-size: 0.95rem;
            margin-bottom: 1.2rem;
            color: var(--text);
            font-weight: 600;
        }
        .attribution a {
            color: var(--accent);
            text-decoration: none;
        }
        .attribution a:hover {
            text-decoration: underline;
        }
        .grid {
            display: grid;
            grid-template-columns: 1.25fr 0.75fr;
            gap: 1.5rem;
        }
        @media (max-width: 860px) {
            .grid { grid-template-columns: 1fr; }
        }
        .form-group {
            margin-bottom: 1.2rem;
        }
        label {
            display: block;
            font-weight: 600;
            margin-bottom: 0.45rem;
            color: var(--subtext);
            font-size: 0.9rem;
        }
        textarea, input[type="text"], input[type="number"], input[type="file"], select {
            width: 100%;
            box-sizing: border-box;
            background: #091024;
            border: 1px solid var(--border);
            color: var(--text);
            padding: 0.75rem;
            border-radius: 8px;
            font-size: 0.95rem;
        }
        textarea:focus, input:focus, select:focus {
            outline: 1px solid var(--accent);
            box-shadow: 0 0 10px var(--accent-glow);
        }
        .vocal-events-bar {
            margin-top: 0.5rem;
            margin-bottom: 0.8rem;
            background: #091024;
            border: 1px solid var(--border);
            border-radius: 8px;
            padding: 0.6rem 0.8rem;
        }
        .vocal-events-title {
            font-size: 0.75rem;
            color: var(--subtext);
            margin-bottom: 0.4rem;
            display: flex;
            justify-content: space-between;
        }
        .event-buttons {
            display: flex;
            flex-wrap: wrap;
            gap: 0.35rem;
        }
        .tag-btn {
            background: var(--tag-bg);
            border: 1px solid var(--border);
            color: #cbd5e1;
            font-size: 0.75rem;
            padding: 0.25rem 0.55rem;
            border-radius: 6px;
            cursor: pointer;
            transition: all 0.15s;
            width: auto;
            font-weight: 500;
        }
        .tag-btn:hover {
            background: var(--accent);
            color: #0b1329;
            border-color: var(--accent);
        }
        button.main-btn {
            width: 100%;
            background: var(--accent);
            color: #0b1329;
            border: none;
            padding: 0.9rem;
            font-size: 1.05rem;
            font-weight: bold;
            border-radius: 8px;
            cursor: pointer;
            transition: opacity 0.2s;
        }
        button.main-btn:hover { opacity: 0.9; }
        button.main-btn:disabled { background: var(--border); cursor: not-allowed; color: var(--subtext); }
        .audio-preview {
            margin-top: 1.5rem;
            padding: 1.2rem;
            background: #091024;
            border-radius: 8px;
            border: 1px solid var(--border);
        }
        audio { width: 100%; margin-top: 0.5rem; }
        .status {
            margin-top: 1rem;
            color: var(--subtext);
            font-size: 0.85rem;
            word-break: break-word;
            white-space: pre-wrap;
            max-height: 140px;
            overflow-y: auto;
            background: #091024;
            padding: 0.75rem;
            border-radius: 8px;
            border: 1px solid var(--border);
        }
        .gallery {
            margin-top: 2rem;
            border-top: 1px solid var(--border);
            padding-top: 1rem;
        }
        .history-list {
            max-height: 220px;
            overflow-y: auto;
            margin-top: 0.5rem;
        }
        .history-item {
            display: flex;
            justify-content: space-between;
            align-items: center;
            background: #091024;
            padding: 0.5rem 0.75rem;
            margin-bottom: 0.5rem;
            border-radius: 6px;
            border: 1px solid var(--border);
        }
        .history-item span {
            font-size: 0.85rem;
            color: var(--subtext);
            max-width: 50%;
            overflow: hidden;
            text-overflow: ellipsis;
            white-space: nowrap;
        }
        .mode-section {
            display: none;
            border: 1px dashed var(--border);
            padding: 1.2rem;
            border-radius: 8px;
            margin-bottom: 1.2rem;
            background: rgba(9, 16, 36, 0.6);
        }
        .mode-section.active {
            display: block;
        }
        .help-text {
            font-size: 0.8rem;
            color: var(--subtext);
            margin-top: 0.35rem;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Breeze-TTS-2</h1>
        <div class="attribution">
            🔴 Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank">AI With Chucky</a>
        </div>
        <form id="ttsForm" onsubmit="event.preventDefault(); return false;">
            <div class="grid">
                <div>
                    <div class="form-group">
                        <label>Select Core Mode</label>
                        <select id="tts_mode" name="tts_mode" onchange="toggleTtsMode()">
                            <option value="voice_design">🎨 Voice Design (Prompt-Only, No Reference Audio)</option>
                            <option value="voice_clone">🧬 Voice Clone (Timbre, Rhythm & Style Cloning)</option>
                            <option value="voice_direction">🎛️ Voice Direction (Clone + Emotional/Pacing Steering)</option>
                        </select>
                    </div>

                    <div class="form-group">
                        <label id="text_label">Text to Synthesize (English or 中文)</label>
                        <textarea id="text" name="text" rows="4" required>(sigh) It has been quite a journey, but we made it! (laugh)</textarea>

                        <!-- Vocal Events Insertion Toolbar -->
                        <div class="vocal-events-bar">
                            <div class="vocal-events-title">
                                <span>🎭 Expressive Vocal Events (Click to insert):</span>
                                <span>Parentheses for EN / Brackets for 中文</span>
                            </div>
                            <div class="event-buttons">
                                <button type="button" class="tag-btn" onclick="insertTag('(laugh)')">(laugh)</button>
                                <button type="button" class="tag-btn" onclick="insertTag('(sigh)')">(sigh)</button>
                                <button type="button" class="tag-btn" onclick="insertTag('(gasp)')">(gasp)</button>
                                <button type="button" class="tag-btn" onclick="insertTag('(cough)')">(cough)</button>
                                <button type="button" class="tag-btn" onclick="insertTag('(clears throat)')">(clears throat)</button>
                                <button type="button" class="tag-btn" onclick="insertTag('(whispers)')">(whispers)</button>
                                <button type="button" class="tag-btn" onclick="insertTag('[笑]')">[笑]</button>
                                <button type="button" class="tag-btn" onclick="insertTag('[叹气]')">[叹气]</button>
                                <button type="button" class="tag-btn" onclick="insertTag('[咳嗽]')">[咳嗽]</button>
                                <button type="button" class="tag-btn" onclick="insertTag('[清嗓子]')">[清嗓子]</button>
                                <button type="button" class="tag-btn" onclick="insertTag('[耳语]')">[耳语]</button>
                                <button type="button" class="tag-btn" onclick="insertTag('[喘息]')">[喘息]</button>
                            </div>
                        </div>
                        <div id="text_help" class="help-text">Creates a distinctive voice from your natural-language description.</div>
                    </div>

                    <!-- Voice Design Section -->
                    <div id="design_section" class="mode-section active">
                        <div class="form-group">
                            <label>Natural-Language Voice Description</label>
                            <textarea id="design_instruction" name="design_instruction" rows="3">A warm, thoughtful young woman with a clear voice and a calm, reflective delivery.</textarea>
                        </div>
                        <div class="form-group">
                            <label>Quick Persona Presets</label>
                            <select id="design_preset" onchange="applyDesignPreset()">
                                <option value="en_female_calm">EN: Warm & Thoughtful Young Woman</option>
                                <option value="en_male_radio">EN: Energetic & Confident Radio Host</option>
                                <option value="en_narrator">EN: Deep & Resonant Cinematic Storyteller</option>
                                <option value="zh_male_calm">中文: 温和成熟男声（沉稳自然）</option>
                                <option value="zh_female_sweet">中文: 亲切甜美女声（清晰生动）</option>
                            </select>
                        </div>
                    </div>

                    <!-- Reference Audio Section (Clone & Direction) -->
                    <div id="clone_section" class="mode-section">
                        <div class="form-group">
                            <label>Upload Reference Audio Clip (.wav / .mp3)</label>
                            <input type="file" id="ref_audio" name="ref_audio" accept="audio/*">
                        </div>
                        <div class="form-group">
                            <label>Exact Transcript of Reference Audio (Optional - Whisper auto-runs if empty)</label>
                            <textarea id="ref_text" name="ref_text" rows="2" placeholder="Transcribe verbatim or leave empty for auto Whisper transcription..."></textarea>
                        </div>
                        <div class="form-group" id="direction_instruction_group" style="display:none;">
                            <label>🎛️ Voice Direction Steering (Tone, Emotion, Pacing, Delivery)</label>
                            <input type="text" id="direction_instruction" name="direction_instruction" placeholder="e.g., Speak slowly with a restrained, serious tone and dramatic pauses.">
                        </div>
                    </div>
                </div>
                <div>
                    <div class="form-group">
                        <label>CFG Scale (Instruction Adherence)</label>
                        <input type="number" id="cfg_scale" name="cfg_scale" value="4.0" step="0.5" min="1.0" max="7.0">
                        <div class="help-text">Official recommended setting: 4.0.</div>
                    </div>
                    <div class="form-group">
                        <label>Pitch-Preserved Speed Multiplier</label>
                        <select name="speed" id="speed">
                            <option value="1.0">1.0x Normal Speed</option>
                            <option value="0.85">0.85x Slower / Deliberate</option>
                            <option value="1.15">1.15x Faster / Brisk</option>
                            <option value="1.3">1.30x Rapid Speech</option>
                        </select>
                    </div>

                    <button type="button" id="submitBtn" class="main-btn" onclick="submitGeneration()">Synthesize Speech</button>
                    <div id="status" class="status">Ready</div>

                    <div class="audio-preview" id="audioContainer" style="display:none;">
                        <label>Generated Audio Output</label>
                        <audio id="audioPlayer" controls></audio>
                        <div style="margin-top: 0.65rem;">
                            <a id="downloadLink" download="breeze_tts_output.wav" style="color: var(--accent); font-size: 0.85rem; text-decoration: none;">⬇️ Download .WAV File</a>
                        </div>
                    </div>
                </div>
            </div>
        </form>

        <div class="gallery">
            <label>Recent Generation History</label>
            <div id="historyList" class="history-list">
                <div style="color: var(--subtext); font-size: 0.85rem; padding: 0.5rem;">No generations yet.</div>
            </div>
        </div>
    </div>
    <script>
        const designPresets = {
            en_female_calm: {
                instruction: "A warm, thoughtful young woman with a clear voice and a calm, reflective delivery.",
                text: "(sigh) Welcome aboard. It has been a long voyage, but your destination is finally within sight."
            },
            en_male_radio: {
                instruction: "A confident, energetic male narrator with an engaging radio host tone and crisp articulation.",
                text: "Good morning! (laugh) We have got an incredible show lined up for you today, so stay tuned!"
            },
            en_narrator: {
                instruction: "A deep, resonant male voice with expressive pauses, gravitas, and cinematic delivery.",
                text: "(clears throat) Beyond the mountains lay a kingdom forgotten by time itself. (whispers) None dared to enter."
            },
            zh_male_calm: {
                instruction: "一位温和成熟的男声，语调沉稳自然，节奏适中，吐字清晰。",
                text: "[叹气] 这一路走来真不容易，不过我们终于做到了！[笑] 谢谢大家一直以来的支持。"
            },
            zh_female_sweet: {
                instruction: "亲切甜美的年轻女声，声音清脆明亮，充满活力与亲和力。",
                text: "早上好呀！[笑] 今天的天气真不错，让我们一起开启充满希望的一天吧！"
            }
        };

        function toggleTtsMode() {
            const mode = document.getElementById('tts_mode').value;
            const textHelp = document.getElementById('text_help');

            document.querySelectorAll('.mode-section').forEach(sec => sec.classList.remove('active'));
            document.getElementById('direction_instruction_group').style.display = 'none';

            if (mode === 'voice_design') {
                document.getElementById('design_section').classList.add('active');
                textHelp.innerText = "Creates a distinctive voice from your natural-language description without reference audio.";
            } else if (mode === 'voice_clone') {
                document.getElementById('clone_section').classList.add('active');
                textHelp.innerText = "Uses reference audio and exact transcript to clone speaker timbre, rhythm, emotion, and style.";
            } else if (mode === 'voice_direction') {
                document.getElementById('clone_section').classList.add('active');
                document.getElementById('direction_instruction_group').style.display = 'block';
                textHelp.innerText = "Preserves reference speaker identity while steering delivery, tone, and pacing with instructions.";
            }
        }

        function applyDesignPreset() {
            const val = document.getElementById('design_preset').value;
            if (designPresets[val]) {
                document.getElementById('design_instruction').value = designPresets[val].instruction;
                document.getElementById('text').value = designPresets[val].text;
            }
        }

        function insertTag(tag) {
            const textarea = document.getElementById('text');
            const start = textarea.selectionStart;
            const end = textarea.selectionEnd;
            const text = textarea.value;
            textarea.value = text.substring(0, start) + tag + text.substring(end);
            textarea.selectionStart = textarea.selectionEnd = start + tag.length;
            textarea.focus();
        }

        async function loadHistory() {
            try {
                const resp = await fetch('/history', {
                    headers: { 'bypass-tunnel-reminder': 'true' }
                });
                const data = await resp.json();
                const historyList = document.getElementById('historyList');
                if (data.length === 0) {
                    historyList.innerHTML = '<div style="color: var(--subtext); font-size: 0.85rem; padding: 0.5rem;">No generations yet.</div>';
                    return;
                }
                historyList.innerHTML = data.map(item => `
                    <div class="history-item">
                        <span>${item.filename}</span>
                        <audio controls src="${item.url}" style="width: 260px; height: 32px;"></audio>
                    </div>
                `).join('');
            } catch(e) {}
        }

        async function pollTaskStatus(taskId) {
            const btn = document.getElementById('submitBtn');
            const status = document.getElementById('status');
            const audioContainer = document.getElementById('audioContainer');
            const audioPlayer = document.getElementById('audioPlayer');
            const downloadLink = document.getElementById('downloadLink');

            const interval = setInterval(async () => {
                try {
                    const resp = await fetch('/task/' + taskId, {
                        headers: { 'bypass-tunnel-reminder': 'true' }
                    });
                    const data = await resp.json();

                    if (data.status === 'processing') {
                        status.innerText = data.stage || 'Synthesizing audio...';
                    } else if (data.status === 'completed') {
                        clearInterval(interval);
                        btn.disabled = false;
                        btn.innerText = 'Synthesize Speech';
                        status.style.color = 'var(--success)';
                        status.innerText = '✅ Synthesis complete!';
                        audioPlayer.src = data.audio_url;
                        downloadLink.href = data.audio_url;
                        audioContainer.style.display = 'block';
                        audioPlayer.play();
                        loadHistory();
                    } else if (data.status === 'failed') {
                        clearInterval(interval);
                        btn.disabled = false;
                        btn.innerText = 'Synthesize Speech';
                        status.style.color = 'var(--error)';
                        status.innerText = 'Error: ' + data.error;
                    }
                } catch (err) {
                    clearInterval(interval);
                    btn.disabled = false;
                    btn.innerText = 'Synthesize Speech';
                    status.style.color = 'var(--error)';
                    status.innerText = 'Polling failed: ' + err.message;
                }
            }, 2000);
        }

        async function submitGeneration() {
            const btn = document.getElementById('submitBtn');
            const status = document.getElementById('status');
            const form = document.getElementById('ttsForm');

            btn.disabled = true;
            btn.innerText = 'Synthesizing...';
            status.style.color = 'var(--subtext)';
            status.innerText = 'Sending generation request to Breeze-TTS-2 engine...';

            const formData = new FormData(form);
            try {
                const resp = await fetch('/generate', {
                    method: 'POST',
                    headers: { 'bypass-tunnel-reminder': 'true' },
                    body: formData
                });

                const data = await resp.json();
                if (!resp.ok || data.error) {
                    status.style.color = 'var(--error)';
                    status.innerText = 'Error: ' + (data.error || 'Server error.');
                    btn.disabled = false;
                    btn.innerText = 'Synthesize Speech';
                } else {
                    pollTaskStatus(data.task_id);
                }
            } catch(err) {
                status.style.color = 'var(--error)';
                status.innerText = 'Request failed: ' + err.message;
                btn.disabled = false;
                btn.innerText = 'Synthesize Speech';
            }
        }

        loadHistory();
    </script>
</body>
</html>
"""

def process_single_synthesis(text, instruction=None, cfg_scale=4.0, ref_audio=None, ref_text=None):
    raw_out = os.path.join(OUTPUT_DIR, f"temp_{uuid.uuid4().hex[:8]}.wav")
    model_path = "/content/Breeze-TTS-2"

    # Handle Auto-Transcription via Whisper
    if ref_audio and os.path.exists(ref_audio):
        if not ref_text or not ref_text.strip():
            print(f"[WHISPER] Transcribing reference clip {ref_audio}...", flush=True)
            result = whisper_model.transcribe(ref_audio)
            ref_text = result["text"].strip()
            print(f"[WHISPER] Transcribed text: {ref_text}", flush=True)

    # Fast Path: In-memory python engine execution
    if tts_engine is not None:
        try:
            inst = instruction.strip() if instruction and instruction.strip() else ("A natural, clear speaker." if not ref_audio else None)
            tts_engine.infer(
                text=text,
                output_path=raw_out,
                instruction=inst,
                cfg_scale=cfg_scale,
                ref_audio=ref_audio,
                ref_text=ref_text
            )
            return raw_out
        except Exception as err:
            print(f"[INFER DIRECT ERROR] {err}. Falling back to CLI subprocess call...", flush=True)

    # Fallback CLI execution
    cmd = [sys.executable, "infer.py", model_path, "--text", text, "--output", raw_out]
    if ref_audio and os.path.exists(ref_audio) and ref_text:
        cmd.extend(["--ref-audio", ref_audio, "--ref-text", ref_text])

    if instruction and instruction.strip():
        cmd.extend(["--instruction", instruction.strip(), "--cfg-scale", str(cfg_scale)])
    elif not ref_audio:
        cmd.extend(["--instruction", "A natural, clear speaker.", "--cfg-scale", "4.0"])

    print(f"[INFER] Executing: {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd, cwd="/content/breeze-tts", capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"[INFER ERROR] {proc.stderr or proc.stdout}", flush=True)
        raise RuntimeError(proc.stderr or proc.stdout or "CLI inference failed.")
    return raw_out

def apply_speed_modification(input_wav, speed_factor):
    if abs(speed_factor - 1.0) < 0.01:
        return
    temp_speed = input_wav.replace(".wav", "_speed.wav")
    cmd = ["ffmpeg", "-y", "-i", input_wav, "-filter:a", f"atempo={speed_factor}", "-vn", temp_speed]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.replace(temp_speed, input_wav)

def run_inference_worker(task_id, form_data, files_data):
    try:
        tts_mode = form_data.get('tts_mode', 'voice_design')
        text = form_data.get('text', '').strip()
        cfg_scale = float(form_data.get('cfg_scale', '4.0'))
        speed_factor = float(form_data.get('speed', '1.0'))

        print(f"[TASK {task_id}] Running mode '{tts_mode}'...", flush=True)
        out_filename = f"output_{uuid.uuid4().hex[:8]}.wav"
        final_out_path = os.path.join(OUTPUT_DIR, out_filename)

        ref_audio_path = files_data.get('ref_audio_path')

        if tts_mode == 'voice_design':
            tasks[task_id]['stage'] = "Designing voice from natural-language prompt..."
            instruction = form_data.get('design_instruction', 'A warm, clear speaker.')
            seg_path = process_single_synthesis(text, instruction=instruction, cfg_scale=cfg_scale)
            os.replace(seg_path, final_out_path)

        elif tts_mode == 'voice_clone':
            tasks[task_id]['stage'] = "Cloning voice timbre and delivery from reference audio..."
            ref_text = form_data.get('ref_text', '').strip()
            seg_path = process_single_synthesis(text, ref_audio=ref_audio_path, ref_text=ref_text)
            os.replace(seg_path, final_out_path)

        elif tts_mode == 'voice_direction':
            tasks[task_id]['stage'] = "Applying voice direction and steering delivery..."
            ref_text = form_data.get('ref_text', '').strip()
            dir_inst = form_data.get('direction_instruction', 'A calm, deliberate delivery.')
            seg_path = process_single_synthesis(text, instruction=dir_inst, cfg_scale=cfg_scale, ref_audio=ref_audio_path, ref_text=ref_text)
            os.replace(seg_path, final_out_path)

        # Pitch-preserved speed adjustment
        if speed_factor != 1.0 and os.path.exists(final_out_path):
            tasks[task_id]['stage'] = "Applying speed multiplier..."
            apply_speed_modification(final_out_path, speed_factor)

        print(f"[TASK {task_id}] Synthesis finished: {out_filename}", flush=True)
        tasks[task_id] = {'status': 'completed', 'audio_url': f'/audio/{out_filename}'}
    except Exception as e:
        err_msg = traceback.format_exc()
        print(f"[TASK ERROR] {err_msg}", flush=True)
        tasks[task_id] = {'status': 'failed', 'error': str(e)}

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/history', methods=['GET'])
def history():
    files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.wav') and not f.startswith('temp_')]
    files.sort(key=lambda x: os.path.getmtime(os.path.join(OUTPUT_DIR, x)), reverse=True)
    history_data = [{'filename': f, 'url': f'/audio/{f}'} for f in files[:10]]
    return jsonify(history_data)

@app.route('/generate', methods=['POST'])
def generate():
    try:
        task_id = uuid.uuid4().hex[:8]
        tasks[task_id] = {'status': 'processing', 'stage': 'Initializing synthesis task...'}

        form_data = request.form.to_dict()
        files_data = {}

        fobj = request.files.get('ref_audio')
        if fobj and fobj.filename != '':
            saved_path = os.path.join(UPLOAD_DIR, f"{uuid.uuid4().hex[:6]}_{fobj.filename}")
            fobj.save(saved_path)
            files_data['ref_audio_path'] = saved_path

        t = threading.Thread(target=run_inference_worker, args=(task_id, form_data, files_data))
        t.start()

        return jsonify({'task_id': task_id})
    except Exception as e:
        print(f"[GENERATE ERROR] {traceback.format_exc()}", flush=True)
        return jsonify({'error': str(e)}), 500

@app.route('/task/<task_id>', methods=['GET'])
def get_task(task_id):
    task = tasks.get(task_id)
    if not task:
        return jsonify({'status': 'failed', 'error': 'Task not found'}), 404
    return jsonify(task)

@app.route('/audio/<filename>')
def serve_audio(filename):
    return send_from_directory(OUTPUT_DIR, filename)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)


In [ ]:
#@title Step 3B: Start Server
import subprocess
import threading
import time
import urllib.request
from IPython.display import display, HTML

print("🚀 Starting Flask Server in /content/breeze-tts...")

# 1. Start Flask backend with unbuffered stdout & stderr streamed directly to logs
flask_process = subprocess.Popen(
    ['python3', '-u', 'app.py'],
    cwd='/content/breeze-tts',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def stream_flask_logs():
    for line in iter(flask_process.stdout.readline, ''):
        if line.strip():
            print(f"[FLASK] {line.strip()}", flush=True)

threading.Thread(target=stream_flask_logs, daemon=True).start()
time.sleep(3)

# 2. Launch Localtunnel
print("🌐 Initializing Localtunnel connection...")
lt_process = subprocess.Popen(
    ["npx", "localtunnel", "--port", "5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

lt_url = ""
for line in iter(lt_process.stdout.readline, ''):
    print(f"[TUNNEL] {line.strip()}", flush=True)
    if "your url is:" in line:
        lt_url = line.split("your url is:")[1].strip()
        break

def stream_tunnel_logs():
    for line in iter(lt_process.stdout.readline, ''):
        if line.strip():
            print(f"[TUNNEL] {line.strip()}", flush=True)

threading.Thread(target=stream_tunnel_logs, daemon=True).start()

try:
    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
except Exception:
    endpoint_ip = "Unavailable"

display(HTML(f'''
    <div style="padding: 16px; background-color: #152243; border-radius: 8px; color: #f8fafc; font-family: sans-serif; border: 1px solid #233563; margin-top: 10px;">
        <h3 style="color: #38bdf8; margin-top: 0;">🚀 Breeze-TTS-2 is Online</h3>
        <p style="margin-bottom: 0.8rem;">🔴 <strong>Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" style="color: #38bdf8;">AI With Chucky</a></strong></p>
        <p><strong>Tunnel URL:</strong> <a href="{lt_url}" target="_blank" style="color: #38bdf8; font-weight: bold; font-size: 1.1rem;">{lt_url}</a></p>
        <p><strong>Tunnel Password (Colab Public IP):</strong> <code style="background: #091024; padding: 4px 8px; border-radius: 4px; color: #4ade80;">{endpoint_ip}</code></p>
        <p style="font-size: 0.85rem; color: #94a3b8; margin-bottom: 0;"><em>Note: Open the Tunnel URL, paste the Colab Public IP if prompted, and click 'Click to Submit'.</em></p>
    </div>
'''))

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Stopping servers...")
    flask_process.terminate()
    lt_process.terminate()

---
### 📦 Breeze-TTS-2 Pro Workstation

For the extended production workstation—featuring multi-speaker podcast generation, paragraph-chunked audiobook synthesis, dynamic emotional performance modes, commercial voiceover profiles, and regional accent localization—refer to the **[Breeze-TTS-2 Pro Notebook](https://www.patreon.com/AiWithChucky/posts/breeze-tts-2-pro-169600516)**.